In [55]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import random_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [64]:
dataset_path = "dataset/pomegranate_diseases_dataset"
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])


dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

In [ ]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size


train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

4079

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [77]:
model = nn.Sequential(
    nn.Conv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        in_channels=16,
        out_channels=32,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(32*32*32,100),
    nn.ReLU(),

    nn.Linear(100,32),
    nn.ReLU(),

    nn.Linear(32,5)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

In [ ]:
def train(x,y):
    output = model(x)
    loss = criterion(output,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [108]:
epoch = 5
for k in range(epoch):
    batches = 0
    imgs = 0
    total_loss = 0
    for images,labels in train_loader:
        x = images
        y = labels
        total_loss = total_loss + train(x,y)
        batches = batches + 1
        imgs = imgs + 32
        print("\rImages:",imgs,end="")
    
    print(f"\nEpoch : {k+1} || Loss : {total_loss/(batches)}")

Images: 2944

KeyboardInterrupt: 

In [105]:
image, label = test_dataset[1]

model.eval()

with torch.no_grad():
    output = model(image.unsqueeze(0))
    prediction = torch.argmax(output, dim=1)

print("Actual:", test_dataset.dataset.classes[label])
print("Predicted:", test_dataset.dataset.classes[prediction.item()])

Actual: Alternaria
Predicted: Healthy


In [107]:
from collections import Counter

counts = Counter(dataset.targets)

for class_name, class_idx in dataset.class_to_idx.items():
    print(class_name, counts[class_idx])
model.eval()

prediction_counts = Counter()

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        predictions = torch.argmax(output, dim=1)

        for prediction in predictions:
            prediction_counts[prediction.item()] += 1

for class_name, class_idx in test_dataset.dataset.class_to_idx.items():
    print(class_name, prediction_counts[class_idx])

Alternaria 886
Anthracnose 1166
Bacterial_Blight 966
Cercospora 631
Healthy 1450
Alternaria 0
Anthracnose 0
Bacterial_Blight 0
Cercospora 0
Healthy 1020
